# M17 - Vision v3: stars and irregular polygons
A star/octagon test of the live v2 "eyes" (`outputs/m17/diag_v2.json`) showed what is really weak:
5-point stars and regular octagons are already 100%, but **6-point stars are 0% on sides** (it always says 10)
and **irregular octagons are 23% on sides** (it says 7 / heptagon). The v2 training mix was 75% 5-point stars
and had few lopsided octagons.

This notebook continues training the v2 vision tower (language model frozen, so tutoring cannot change) on a
balanced set: stars with 4-8 points and irregular hexagons, heptagons and octagons, mixed with a replay of v2's
own data so nothing else is forgotten. v3 is served on :8081 next to the live v2 and compared on every v2 test
plus the star/octagon test; the live app is switched only if v3 wins on stars/octagons with no regression.
Children's data stays under `data/raw/` (git-ignored).


In [ ]:
# ---- setup: reuse notebook 16's helpers (shape drawing, row format, evaluation) ----
import json, os, sys, time, math, random, re, shutil, subprocess
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "PROJECT_BRIEF.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
_nb16 = json.load(open(ROOT / "notebooks/16_vision_v2.ipynb"))
for _i in (1, 2):                                   # setup + shape generator (its own step A is already done)
    exec("".join(_nb16["cells"][_i]["source"]))
V2_DIR = OUT                                        # outputs/m16 (v2 training output)
V2_GEN = GEN                                        # data/raw/vision_v2
OUT = ROOT / "outputs" / "m17"; OUT.mkdir(parents=True, exist_ok=True)
GEN3 = ROOT / "data" / "raw" / "vision_v3"; GEN3.mkdir(parents=True, exist_ok=True)
V3_HF = ROOT / "models" / "geeta-4b-v3-hf"
MMPROJ_V3 = GGUF_DIR / "mmproj-geeta-4b-v3-f16.gguf"
def done(f): return (OUT / f".{f}.done").exists()
def mark(f, p=None): (OUT / f".{f}.done").write_text(json.dumps({"ts": time.time(), "p": p}))
def log(name, obj): (OUT / name).write_text(json.dumps(obj, indent=1, default=str)); print("[m17] wrote", name)
print("root", ROOT)

In [ ]:
# ---- A: balanced stars (4-8 points) + irregular 6/7/8-gons, exact labels ----
from PIL import Image, ImageDraw
def star_pts(p, inner, rot):
    return [((1 if i % 2 == 0 else inner) * math.cos(rot + math.pi * i / p),
             (1 if i % 2 == 0 else inner) * math.sin(rot + math.pi * i / p)) for i in range(2 * p)]

def irregular(n, rng):
    while True:
        angs = sorted(rng.uniform(0, 2 * math.pi) for _ in range(n))
        if min((angs[(i + 1) % n] - angs[i]) % (2 * math.pi) for i in range(n)) < 0.35:
            continue
        pts = [(rng.uniform(0.7, 1) * math.cos(t), rng.uniform(0.7, 1) * math.sin(t)) for t in angs]
        if _clear_corners(pts):
            return pts

NAMES = {6: "hexagon", 7: "heptagon", 8: "octagon"}
def target_shape(kind, rng):
    if kind.startswith("star"):
        p = int(kind[4:])
        return star_pts(p, rng.uniform(0.38, 0.55), -math.pi / 2 if rng.random() < .6 else rng.uniform(0, 2 * math.pi)), \
            dict(sides=2 * p, name="star", sym=p)
    n = int(kind[3:])
    if kind.startswith("reg"):
        return _reg(n, rng.uniform(0, 2 * math.pi)), dict(sides=n, name=NAMES[n], sym=n)
    return irregular(n, rng), dict(sides=n, name=NAMES[n], sym=None)

PLAN = {"star4": 70, "star5": 70, "star6": 130, "star7": 100, "star8": 100,
        "irr6": 150, "irr7": 150, "irr8": 260, "reg6": 25, "reg7": 25, "reg8": 20}

def one(kind, rng, path):
    S = rng.randint(288, 448); im = _canvas(rng, S, S); dr = ImageDraw.Draw(im)
    shape, info = target_shape(kind, rng)
    ink = rng.choice([(20, 20, 20), (60, 60, 60), (90, 90, 95), (25, 35, 110)])
    fill = rng.choice([None] * 7 + [(40, 40, 40), (120, 120, 120), (200, 200, 200)])
    draw_shape(dr, shape, S / 2 + rng.uniform(-S * .06, S * .06), S / 2 + rng.uniform(-S * .06, S * .06),
               S * rng.uniform(.27, .4), rng, ink, rng.randint(2, 6), fill)
    _finish(im, rng, path); n = info["sides"]
    qa = [(Q_SIDES, str(n)), (Q_CORN, str(n)), (Q_SEG, f"segments: {n}, angles: {n}"), (Q_NAME, info["name"]),
          (Q_DESC, f"{info['name']}: {n} sides, {n} corners")]
    if info["sym"] is not None:
        qa.append((Q_SYM, str(info["sym"])))
    return qa

if done("A"):
    print("A done")
else:
    rng = random.Random(1701); d = GEN3 / "shapes"; d.mkdir(parents=True, exist_ok=True); rows = []
    for kind, k in PLAN.items():
        for j in range(k):
            p = d / f"{kind}_{j:04d}.jpg"; qa = one(kind, rng, p)
            # sides/corners are what failed: always ask one of them, plus one other question
            first = rng.choice([q for q in qa if q[0] in (Q_SIDES, Q_CORN)])
            other = rng.choice([q for q in qa if q is not first])
            rows += [row(p.relative_to(ROOT), q, a) for q, a in (first, other)]
    (GEN3 / "targeted_train.jsonl").write_text("\n".join(json.dumps(r) for r in rows) + "\n")
    mark("A", {"images": sum(PLAN.values()), "rows": len(rows)}); print("targeted rows", len(rows))

In [ ]:
# ---- B: replay of v2's own data, so nothing it learned is forgotten ----
if done("B"):
    print("B done")
else:
    rng = random.Random(1702)
    def take(f, k):
        rr = [json.loads(l) for l in open(f) if l.strip()]
        return rr if k >= len(rr) else rng.sample(rr, k)
    replay = (take(V2_GEN / "child_train.jsonl", 288) + take(V2_GEN / "pencil_train.jsonl", 320)
              + take(ROOT / "data/splits/train_vision.jsonl", 320) + take(V2_GEN / "shapes_train.jsonl", 420))
    (GEN3 / "replay_train.jsonl").write_text("\n".join(json.dumps(r) for r in replay) + "\n")
    mark("B", {"rows": len(replay)}); print("replay rows", len(replay))

In [ ]:
# ---- C: continue training the v2 vision tower (language model frozen) ----
if done("C"):
    print("C done")
else:
    import mlx.core as mx, mlx.nn as nn, mlx.optimizers as optim, mlx_vlm
    from mlx.utils import tree_flatten
    from mlx_vlm import utils as vutils
    from mlx_vlm.trainer.utils import grad_checkpoint, get_learning_rate
    rows = [json.loads(l) for f in ("targeted_train.jsonl", "replay_train.jsonl") for l in open(GEN3 / f) if l.strip()]
    random.Random(1703).shuffle(rows)
    model, processor = vutils.load(MLX_DIR)
    model.vision_tower.load_weights(list(mx.load(str(V2_DIR / "vision_tower_v2" / "adapters.safetensors")).items()))
    model.train()
    MAXSIDE = 1100
    def prep(r):
        msgs = r["messages"]; img = Image.open(ROOT / msgs[0]["content"][0]["path"]).convert("RGB")
        if max(img.size) > MAXSIDE:
            s = MAXSIDE / max(img.size); img = img.resize((int(img.width * s), int(img.height * s)))
        pf = mlx_vlm.apply_chat_template(processor, model.config, msgs, add_generation_prompt=False, num_images=1)
        pp = mlx_vlm.apply_chat_template(processor, model.config, msgs[:1], add_generation_prompt=True, num_images=1)
        a = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[pf]); b = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[pp])
        off = b["input_ids"].shape[1]
        assert bool((a["input_ids"][:, :off] == b["input_ids"]).all()), "prefix misaligned"
        return a["input_ids"], a.get("pixel_values"), a.get("image_grid_thw"), off
    model.freeze(); model.vision_tower.unfreeze(); grad_checkpoint(model.vision_tower.blocks[0])
    opt = optim.Adam(learning_rate=5e-6)            # half of v2's rate: a gentle top-up, not a re-train
    def loss_fn(m_, ids, pv, grid, off):
        logits = m_(ids, pv, None, None, image_grid_thw=grid).logits[:, :-1]; tg = ids[:, 1:]
        steps = mx.arange(1, tg.shape[1] + 1); mask = steps >= off
        ce = nn.losses.cross_entropy(logits, tg) * mask
        return ce.astype(mx.float32).sum() / mx.maximum(mask.sum(), 1)
    vg = nn.value_and_grad(model, loss_fn)
    ITERS = len(rows); hist = []; t0 = time.time(); skipped = 0
    for it in range(1, ITERS + 1):
        opt.learning_rate = get_learning_rate(ITERS, it, 50, 5e-6, 5e-7)
        try:
            ids, pv, grid, off = prep(rows[it - 1])
        except Exception:
            skipped += 1; continue
        lv, g = vg(model, ids, pv, grid, off); opt.update(model, g); mx.eval(model.parameters(), opt.state)
        hist.append(float(lv))
        if it % 50 == 0 or it == 1:
            el = time.time() - t0
            msg = f"[vision v3] {it}/{ITERS} loss {sum(hist[-50:]) / len(hist[-50:]):.4f} | {el / 60:.1f} min, eta {el / it * (ITERS - it) / 60:.0f} min"
            print(msg, flush=True)
            with open(OUT / "progress.txt", "a") as pf: pf.write(msg + "\n")
    vdir = OUT / "vision_tower_v3"; vdir.mkdir(exist_ok=True)
    mx.save_safetensors(str(vdir / "adapters.safetensors"), dict(tree_flatten(model.vision_tower.parameters())))
    n = 50; first, last = sum(hist[:n]) / n, sum(hist[-n:]) / n
    log("train_report.json", {"iters": ITERS, "skipped": skipped, "minutes": round((time.time() - t0) / 60, 1),
                              "first_loss": first, "last_loss": last, "loss_curve": [round(x, 4) for x in hist[::20]]})
    mark("C", {"first": first, "last": last})

In [ ]:
# ---- D: v3 vision weights -> HF dir -> mmproj GGUF (language model unchanged) ----
if done("D"):
    print("D done")
else:
    import mlx.core as mx, numpy as np
    trained = mx.load(str(OUT / "vision_tower_v3" / "adapters.safetensors"))
    base = dict(mx.load(str(V1_HF / "model.safetensors")))
    n_rep = 0
    for k, v in trained.items():
        hk = "model.visual." + k
        a = np.asarray(v.astype(mx.float16))
        if hk.endswith("patch_embed.proj.weight") and a.ndim == 5:
            a = np.transpose(a, (0, 4, 1, 2, 3))
        assert hk in base and tuple(base[hk].shape) == a.shape, f"bad key/shape {hk}"
        base[hk] = mx.array(a.astype(np.float16)); n_rep += 1
    assert n_rep == sum(1 for k in base if k.startswith("model.visual.")), "not every vision tensor was replaced"
    V3_HF.mkdir(parents=True, exist_ok=True)
    mx.save_safetensors(str(V3_HF / "model.safetensors"), base)
    for fn in os.listdir(V1_HF):
        if fn != "model.safetensors": shutil.copy2(V1_HF / fn, V3_HF / fn)
    del base
    r = subprocess.run([sys.executable, str(LLAMA / "convert_hf_to_gguf.py"), str(V3_HF), "--mmproj", "--outtype", "f16",
                        "--outfile", str(MMPROJ_V3)], cwd=LLAMA, capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-800:] if r.returncode else "")
    assert MMPROJ_V3.exists() and MMPROJ_V3.stat().st_size > 100_000_000, "mmproj v3 not produced"
    mark("D", {"mmproj_gb": round(MMPROJ_V3.stat().st_size / 2**30, 3)}); print("mmproj v3", MMPROJ_V3)

In [ ]:
# ---- E: live v2 (:8080) vs v3 (:8081) on every v2 test + the star/octagon test ----
import urllib.request
LAB = {k: v for k, v in json.loads((HW / "reference_labels.json").read_text()).items() if k != "_notes"}
try:
    urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=3)
    raise SystemExit("something is already serving on :8081 - stop it first so v3 is what gets tested")
except OSError:
    pass
exec("".join(_nb16["cells"][7]["source"]).split("if done(\"F\")")[0]
     .replace("MMPROJ_V2", "MMPROJ_V3").replace('OUT / "server_v2.log"', 'OUT / "server_v3.log"'))
if done("E"):
    print(json.loads((OUT / "compare.json").read_text()))
else:
    t0 = time.time()
    r2, r3 = evaluate("http://127.0.0.1:8080"), evaluate(V2URL)
    diag = {}
    for tag, url in (("v2", "http://127.0.0.1:8080"), ("v3", V2URL)):
        out = OUT / f"diag_{tag}_gate.json"
        subprocess.run([sys.executable, str(OUT / "shape_diag.py"), url, str(out)], check=True,
                       env=dict(os.environ, N="30"), capture_output=True)
        diag[tag] = json.loads(out.read_text())
    weak = ["star6|sides", "star6|corners", "oct_irr|sides", "oct_irr|corners", "oct_irr|name"]
    frac = lambda s: int(s.split("/")[0]) / int(s.split("/")[1])
    avg = lambda d: sum(v["acc"] for v in d.values()) / len(d)
    cmp = {"v2_live": r2, "v3": r3, "minutes": round((time.time() - t0) / 60, 1),
           "diag_weak": {k: [diag["v2"][k]["acc"], diag["v3"][k]["acc"]] for k in weak},
           "diag_all_mean": [round(avg(diag["v2"]), 4), round(avg(diag["v3"]), 4)]}
    gate = (all(diag["v3"][k]["acc"] > diag["v2"][k]["acc"] for k in weak)
            and avg(diag["v3"]) > avg(diag["v2"])
            and r3["shapes_generated_test"] >= r2["shapes_generated_test"]
            and r3["real_child_handwriting_exact"] >= r2["real_child_handwriting_exact"] - 0.01
            and r3["synthetic_handwriting_char_acc"] >= r2["synthetic_handwriting_char_acc"] - 0.01
            and frac(r3["real_shapes_sides"]) >= frac(r2["real_shapes_sides"])
            and frac(r3["real_shapes_names"]) >= frac(r2["real_shapes_names"])
            and frac(r3["real_photo_pages_items"]) >= frac(r2["real_photo_pages_items"]))
    cmp["v3_better_no_regression"] = gate
    log("compare.json", cmp); print(json.dumps(cmp, indent=1)); mark("E", gate)
if "srv" in globals():
    srv.terminate()                                  # the :8081 test server